# Week 4 build · Bigram to one attention head

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mjtri/learnml/blob/main/notebooks/week-04.ipynb)

**~3 hours · CPU is enough (GPU only for Part F) · run top to bottom.**

**The one rule:** every experiment is preceded by a 🔮 **predict cell**. Most predictions this week are *shapes*: type the shape you expect at every step, and how sure you are, *before* running. The notebook refuses empty predictions. The gap between guess and result is what you are here to collect (canvas steps 7–8).

**`SMOKE`** in the second cell shrinks the corpus, model and step counts so the whole notebook runs on a CPU in a few minutes. Set it to `False` for the real run.

| Part | Topic | Time |
|---|---|---|
| A | Tokens and the corpus | 20 min |
| B | Bigram by counting, then by gradient descent | 40 min |
| C | Lesson 4's three tokens in `torch` | 25 min |
| D | Position vectors plus one head | 45 min |
| E | Break it three ways | 30 min |
| F | Stretch: real text (GPU recommended) | 20 min |

In [ ]:
#@title Setup: run me first
import numpy as np, torch, matplotlib.pyplot as plt
from IPython.display import Audio, Markdown, display
torch.manual_seed(0); np.random.seed(0)
_LEDGER = {}

def predict(tag, prediction, confidence):
    """Log a prediction BEFORE the experiment. Refuses empty predictions."""
    if not str(prediction).strip():
        raise ValueError(f"[{tag}] Write a prediction first. A rough guess is fine; an empty one is not.")
    _LEDGER[tag] = {"prediction": str(prediction).strip(), "confidence": int(confidence), "actual": "", "surprise": ""}
    print(f"Logged [{tag}] at {confidence}% confidence: {prediction}")

def record(tag, what_happened, surprise="none"):
    if tag not in _LEDGER:
        raise ValueError(f"[{tag}] has no prediction yet. Run its predict cell first.")
    _LEDGER[tag].update(actual=str(what_happened).strip(), surprise=surprise)
    print(f"Recorded [{tag}] surprise={surprise}")

def reveal():
    rows = ["| # | prediction | sure | what happened | surprise |", "|---|---|---|---|---|"]
    rows += [f"| {t} | {r['prediction']} | {r['confidence']}% | {r['actual']} | {r['surprise']} |" for t, r in _LEDGER.items()]
    print("\n".join(rows)); display(Markdown("\n".join(rows)))

print("torch", torch.__version__, "| ready")

In [ ]:
SMOKE = True   # True: tiny corpus / model / steps, runs on a CPU in minutes. False: the real run (GPU helps in Part F).
import math, random
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CFG = dict(lines=400 if SMOKE else 4000, steps=1000 if SMOKE else 3000, d=32 if SMOKE else 64, T=16, B=32 if SMOKE else 64, lr=1e-2 if SMOKE else 5e-3)
print("SMOKE =", SMOKE, "| device:", DEVICE, "|", CFG)

---
## Part A · Tokens and the corpus (20 min)

**Words used in this part.** *token*: the unit the model reads; here one character. *sequence*: an ordered list of tokens. *context length* (`T`, Karpathy's `block_size`): how many previous tokens the model may look at. *ln V*: the loss of a model that spreads probability evenly over `V` tokens, the "knows nothing" line.

The corpus is a synthetic log of haptic cues. Each line is `<hand>:<digits>;<hand>` where the hand letter fixes which digits may appear (`L` 1–3, `R` 4–6, `B` 7–9) and is repeated at the end. Two things in every line depend on a token that is *not* the previous one. Keep that in mind for Part D.

In [ ]:
RANGE = {"L": "123", "R": "456", "B": "789"}          # the opener fixes the digit range ...
def make_line(rng):
    op = rng.choice("LRB")
    digits = [rng.choice(RANGE[op]) for _ in range(rng.randint(2, 5))]
    return f"{op}:{','.join(digits)};{op}\n"           # ... and must be echoed at the end
rng = random.Random(0)
text = "".join(make_line(rng) for _ in range(CFG["lines"]))
print(text[:70])
print(len(text), "characters in", CFG["lines"], "lines")

In [ ]:
#@title 🔮 Predict A1: How many distinct tokens (characters) does this text contain, and what is ln V?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("A1", prediction, confidence)

In [ ]:
chars = sorted(set(text)); V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[int(i)] for i in ids)
data = torch.tensor(encode(text))
print("V =", V, "| tokens:", repr("".join(chars)))
print("data shape", tuple(data.shape), "dtype", data.dtype)
print(f"'knows nothing' line: ln V = {math.log(V):.3f}")
n_train = int(0.9 * len(data)); train, val = data[:n_train], data[n_train:]
print("train", tuple(train.shape), "val", tuple(val.shape))

In [ ]:
#@title 🔮 Predict A2: get_batch returns x and y for B windows of T tokens. What are their shapes, and how does y[0] relate to x[0]?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("A2", prediction, confidence)

In [ ]:
def get_batch(split, B=None, T=None):
    B, T = B or CFG["B"], T or CFG["T"]
    src = train if split == "train" else val
    ix = torch.randint(len(src) - T, (B,))
    x = torch.stack([src[i:i + T] for i in ix])
    y = torch.stack([src[i + 1:i + T + 1] for i in ix])       # the same window, shifted by one token
    return x.to(DEVICE), y.to(DEVICE)
x, y = get_batch("train")
print("x", tuple(x.shape), "y", tuple(y.shape))
print("x[0]:", repr(decode(x[0])))
print("y[0]:", repr(decode(y[0])))

Every position is a labelled example: `y` is `x` shifted by one. A batch of `B × T` tokens is `B × T` training examples at once.

---
## Part B · Bigram: by counting, then by gradient descent (40 min)

**Words used in this part.** *bigram*: predict the next token from the current token only. *next-token prediction*: the training job. *loss*: the surprise `−ln p` at the token that actually came next, averaged (week 3's cross-entropy). *logits*: the raw scores a row of the table holds before the softmax turns them into probabilities.

In [ ]:
#@title 🔮 Predict B1: The counting bigram's loss on val: between ln V and 0, roughly where? After which characters is it most uncertain?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("B1", prediction, confidence)

In [ ]:
N = torch.zeros(V, V)
for a, b in zip(train[:-1].tolist(), train[1:].tolist()):
    N[a, b] += 1                                          # how often b followed a
P = (N + 1) / (N + 1).sum(1, keepdim=True)                # +1 smoothing, then each row sums to 1
count_loss = -P[val[:-1], val[1:]].log().mean()
print(f"counting bigram loss on val = {count_loss:.3f}   (ln V = {math.log(V):.3f})")
print("row sums:", P.sum(1)[:4].tolist(), "...")
for c in ":,;\n":
    top = P[stoi[c]].topk(3)
    print(f"after {c!r}: " + ", ".join(f"{itos[i.item()]!r} {p:.2f}" for p, i in zip(top.values, top.indices)))

In [ ]:
#@title ✍️ B1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("B1", what_happened, surprise)

After `:` the bigram spreads its bets over all nine digits, and after `;` over all three letters. It cannot see the opener. Everything else it gets nearly right. Now the same table, learned:

In [ ]:
#@title 🔮 Predict B2: Bigram as nn.Embedding(V, V): what shape are the logits for x of shape (B, T)? After training, will its loss beat the counting version?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("B2", prediction, confidence)

In [ ]:
import torch.nn as nn, torch.nn.functional as F
class Bigram(nn.Module):
    def __init__(self, V):
        super().__init__()
        self.table = nn.Embedding(V, V)                       # row a = scores for what follows a
    def forward(self, idx, targets=None):
        logits = self.table(idx)                              # (B, T) -> (B, T, V): a lookup, nothing else
        loss = None if targets is None else F.cross_entropy(logits.reshape(-1, V), targets.reshape(-1))
        return logits, loss
bigram = Bigram(V).to(DEVICE)
logits, loss = bigram(x, y)
print("logits", tuple(logits.shape), f"| untrained loss {loss.item():.3f}  (ln V = {math.log(V):.3f})")

In [ ]:
@torch.no_grad()
def eval_loss(model, split="val", iters=50):
    model.eval(); tot = 0.0
    for _ in range(iters):
        xb, yb = get_batch(split); tot += model(xb, yb)[1].item()
    model.train(); return tot / iters

def train_model(model, steps=None, lr=None, log_every=100):
    steps, lr = steps or CFG["steps"], lr or CFG["lr"]
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for step in range(steps):
        xb, yb = get_batch("train")
        _, loss = model(xb, yb)                               # forward + loss
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()   # backward + update
        if step % log_every == 0 or step == steps - 1:
            print(f"step {step:5d}  train {loss.item():.3f}  val {eval_loss(model):.3f}")
    return model
train_model(bigram, lr=0.1)      # a lookup table can take big steps: every row is independent
print(f"trained bigram val loss {eval_loss(bigram):.3f}   vs counting {count_loss:.3f}")

In [ ]:
#@title ✍️ B2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("B2", what_happened, surprise)

In [ ]:
@torch.no_grad()
def generate(model, n=40, T=None):
    T = T or CFG["T"]
    idx = torch.tensor([[stoi["\n"]]], device=DEVICE)         # start right after a line break
    for _ in range(n):
        logits, _ = model(idx[:, -T:])                           # only the last T tokens fit the model
        probs = F.softmax(logits[:, -1, :], dim=-1)             # (1, V): the next-token distribution
        idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)   # draw one token, append, repeat
    return decode(idx[0, 1:])
print(generate(bigram))

Read the generated lines against the rule: digits outside the opener's range, closing letters that do not match. Local structure (`:` after a letter, `,` between digits) is fine. That is exactly what a context length of 1 can and cannot do.

---
## Part C · Lesson 4's three tokens in `torch` (25 min)

**Words used in this part.** *query / key / value*: what a token asks for, offers, and hands over. *scaled dot-product*: scores `Q @ K.T`, divided by `√d`, softmax per row, then `weights @ V`. *causal mask*: scores pointing at later tokens set to `−inf` before the softmax.

Tokens "strong", "left", "pulse", two numbers each, the exact table from lesson 3. Do the "pulse" row on paper first.

In [ ]:
#@title 🔮 Predict C1: Q, K, V are each (3, 2). What shape is Q @ K.T? What is the 'pulse' row of weights after ÷√2 and softmax (three numbers)?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("C1", prediction, confidence)

In [ ]:
Q = torch.tensor([[0., 1], [1, 0], [2, 0]])     # strong, left, pulse: what each asks for
K = torch.tensor([[2., 0], [0, 2], [1, 1]])     # what each offers
Vv = torch.tensor([[3., 0], [0, 3], [1, 1]])    # what each hands over
scores = Q @ K.T
print("scores", tuple(scores.shape)); print(scores)
weights = F.softmax(scores / math.sqrt(2), dim=-1)
print("weights (each row sums to 1):"); print(weights.round(decimals=2))
out = weights @ Vv
print("out", tuple(out.shape)); print(out.round(decimals=2))
ref = F.scaled_dot_product_attention(Q, K, Vv)  # PyTorch's fused version: same numbers
print("matches F.scaled_dot_product_attention:", torch.allclose(out, ref, atol=1e-5))

In [ ]:
#@title ✍️ C1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("C1", what_happened, surprise)

In [ ]:
#@title 🔮 Predict C2: Add the causal mask. Which rows change, and what do rows 1 and 2 ('strong', 'left') become?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("C2", prediction, confidence)

In [ ]:
tril = torch.tril(torch.ones(3, 3))
masked = scores.masked_fill(tril == 0, float("-inf"))   # the future -> -inf -> weight exactly 0
print(masked)
w_masked = F.softmax(masked / math.sqrt(2), dim=-1)
print("masked weights:"); print(w_masked.round(decimals=2))
print("without ÷√2, row 3 would be:", F.softmax(scores[2], dim=-1).round(decimals=2).tolist(), "(sharper)")
ref = F.scaled_dot_product_attention(Q, K, Vv, is_causal=True)
print("matches is_causal=True:", torch.allclose(w_masked @ Vv, ref, atol=1e-5))

Row 1 can only see itself, so its weight is 1.0 whatever the scores were. Row 3 is unchanged: it already saw only the past. The rows still sum to 1 because the softmax renormalises over whatever is left.

---
## Part D · Position vectors plus one head (45 min)

**Words used in this part.** *head size* (`hs`): numbers per head; here equal to `d`, one head. *position embedding* (`wpe`): a `(T, d)` table added to the token embedding so the head can tell first from third. *attention weights*: the `(B, T, T)` matrix the head computes; we keep the last one to look at.

This is the `Head` from Karpathy's *Let's build GPT*, with two switches (`mask`, `scale`) that Part E flips off.

In [ ]:
class Head(nn.Module):
    def __init__(self, d, hs, T, mask=True, scale=True):
        super().__init__()
        self.key, self.query, self.value = (nn.Linear(d, hs, bias=False) for _ in range(3))
        self.register_buffer("tril", torch.tril(torch.ones(T, T)))
        self.mask, self.scale = mask, scale
        self.last = None                                          # the weights, kept for inspection
    def forward(self, x):
        B, T, d = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)       # each (B, T, hs)
        wei = q @ k.transpose(-2, -1)                              # (B, T, T) scores
        if self.scale: wei = wei * k.shape[-1] ** -0.5             # ÷ √hs
        if self.mask: wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1); self.last = wei.detach()
        return wei @ v                                             # (B, T, hs) weighted mix of values

In [ ]:
#@title 🔮 Predict D1: x is (B, T). Shapes of: tok_emb, pos_emb, their sum, the head output, the logits, the attention weights?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D1", prediction, confidence)

In [ ]:
class LM(nn.Module):
    def __init__(self, V, d, T, use_pos=True, mask=True, scale=True):
        super().__init__()
        self.wte = nn.Embedding(V, d)                             # token table    (V, d)
        self.wpe = nn.Embedding(T, d)                             # position table (T, d)
        self.head = Head(d, d, T, mask, scale)
        self.lm_head = nn.Linear(d, V)
        self.use_pos = use_pos
    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.wte(idx)                                       # (B, T, d)
        pos = self.wpe(torch.arange(T, device=idx.device))        # (T, d)
        x = tok + pos if self.use_pos else tok                    # broadcast over B
        logits = self.lm_head(self.head(x))                       # (B, T, V)
        loss = None if targets is None else F.cross_entropy(logits.reshape(-1, V), targets.reshape(-1))
        return logits, loss

In [ ]:
torch.manual_seed(0)
model = LM(V, CFG["d"], CFG["T"]).to(DEVICE)
xb, yb = get_batch("train")
tok = model.wte(xb); pos = model.wpe(torch.arange(CFG["T"], device=DEVICE))
h = model.head(tok + pos); logits, loss = model(xb, yb)
print("x", tuple(xb.shape), "-> tok_emb", tuple(tok.shape), "+ pos_emb", tuple(pos.shape), "->", tuple((tok + pos).shape))
print("head out", tuple(h.shape), "-> logits", tuple(logits.shape), "| attention weights", tuple(model.head.last.shape))
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}   untrained loss {loss.item():.3f}")

In [ ]:
#@title ✍️ D1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D1", what_happened, surprise)

In [ ]:
#@title 🔮 Predict D2: After training: val loss vs the bigram? Which contexts (after ':', ',', ';', newline) improve most, and which cannot improve at all?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D2", prediction, confidence)

In [ ]:
train_model(model)
@torch.no_grad()
def loss_by_context(m, n_batches=20):
    tot, cnt = torch.zeros(V), torch.zeros(V)
    for _ in range(n_batches):
        xb, yb = get_batch("val"); logits, _ = m(xb)
        l = F.cross_entropy(logits.reshape(-1, V), yb.reshape(-1), reduction="none").cpu()
        prev = xb.reshape(-1).cpu()
        tot.index_add_(0, prev, l); cnt.index_add_(0, prev, torch.ones_like(l))
    return tot / cnt.clamp(min=1)
lb, lh = loss_by_context(bigram), loss_by_context(model)
print(f"val loss: bigram {eval_loss(bigram):.3f}   one head {eval_loss(model):.3f}   ln V {math.log(V):.3f}")
print("loss after context     bigram   head")
for c in ":,;\nL":
    print(f"   after {c!r:5}          {lb[stoi[c]]:.2f}   {lh[stoi[c]]:.2f}")

In [ ]:
#@title ✍️ D2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D2", what_happened, surprise)

After `,` the next token is either another digit (still random within the range) or `;`: some of that surprise is real randomness and no model removes it. After `:` and `;` the head can look back at the opener and the loss drops. Now look at *where* it looks:

In [ ]:
#@title 🔮 Predict D3: For the line below: at the ':' (row 1) and the ';' (row 7), where does the weight go: the opener, a digit (which also reveals the hand), or the token itself? And for the second line's ';' (row 15)?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D3", prediction, confidence)

In [ ]:
line = "R:5,4,6;R\nL:2,1;"                              # exactly T = 16 tokens, two openers
ids = torch.tensor([encode(line)], device=DEVICE)
model.eval(); model(ids); W = model.head.last[0].cpu(); model.train()
labels = [c.replace("\n", "⏎") for c in line]
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(W, cmap="Blues"); ax.set_xticks(range(len(line))); ax.set_xticklabels(labels)
ax.set_yticks(range(len(line))); ax.set_yticklabels(labels)
ax.set_xlabel("looks at (key)"); ax.set_ylabel("asking token (query)"); ax.set_title("attention weights, one head"); plt.show()
for i in [1, 7, 15]:
    top = W[i].topk(2)
    print(f"row {i:2d} {labels[i]!r} looks at: " + ", ".join(f"col {j.item():2d} {labels[j.item()]!r} ({w:.2f})" for w, j in zip(top.values, top.indices)))

In [ ]:
#@title ✍️ D3: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D3", what_happened, surprise)

Row 1 has only the opener to look at, and does. Rows 7 and 15 usually pick a *digit* instead: a `5` reveals `R` just as well as the opener does, and the head takes whichever clue is cheapest. Row 15 should look inside the second line, not the first; that is the position table at work. Trained models are full of such shortcuts, and reading the weights is how you find them.

In [ ]:
for _ in range(3):
    print(generate(model, n=30))

The generated lines now respect the opener, most of the time. One head, one table of positions, and the model reads back sixteen tokens. This is the half of a transformer that *routes* information; week 5 adds the half that *processes* it.

---
## Part E · Break it three ways (30 min)

**Words used in this part.** *no mask*: the head may read the future, including the token it is supposed to predict. *no positions*: `use_pos=False`, the tokens before you are a bag. *no √d*: raw scores go straight into the softmax.

Predict the *ranking* of the four validation losses before running, and say which low number is a lie.

In [ ]:
#@title 🔮 Predict E1: Rank the val losses of: full, no positions, no mask, no √d. Which is lowest, and is that a good sign?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("E1", prediction, confidence)

In [ ]:
variants = {"full": dict(), "no positions": dict(use_pos=False), "no mask": dict(mask=False), "no √d": dict(scale=False)}
results = {}
for name, kw in variants.items():
    torch.manual_seed(0)
    m = LM(V, CFG["d"], CFG["T"], **kw).to(DEVICE)
    train_model(m, log_every=10**9)                       # prints only the first and last step
    results[name] = (eval_loss(m, "train"), eval_loss(m), m)
    print(f"  -> {name:13} train {results[name][0]:.3f}   val {results[name][1]:.3f}\n")

In [ ]:
print("no-mask model, generating (at generation time there is no future to read):")
print(generate(results["no mask"][2], n=40))
print("\nfull model:")
print(generate(results["full"][2], n=40))

In [ ]:
#@title ✍️ E1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("E1", what_happened, surprise)

**No mask** scores best on both train and val and generates garbage: during training every position could read its own answer one token ahead through the values, and that shortcut does not exist when generating. This bug is silent; a loss that is *too* good is the only symptom. **No positions** cannot tell the recent opener from an older one when two lines share a window, and cannot count digits. **No √d**: with `d = 32` the raw scores start out several units wide, the softmax is sharp from the first step, and learning is slower; the gap grows with `d` (try `d = 256`).

---
## Part F · Stretch: real text (20 min, GPU recommended, skipped in SMOKE mode)

**Words used in this part.** Nothing new: the same `Bigram` and `LM`, on 1 MB of English with one token per character (`V ≈ 65`, `ln V ≈ 4.17`).

In [ ]:
#@title 🔮 Predict F1: On real English text with character tokens: the bigram's loss, and the one-head model's loss after 3000 steps? (ln V ≈ 4.17)
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("F1", prediction, confidence)

In [ ]:
if SMOKE:
    print("SMOKE mode: skipped. Set SMOKE = False (a GPU helps) to run this part.")
else:
    import urllib.request
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    real = urllib.request.urlopen(url, timeout=30).read().decode()
    chars = sorted(set(real)); V = len(chars)
    stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
    rdata = torch.tensor(encode(real)); n = int(0.9 * len(rdata)); train, val = rdata[:n], rdata[n:]
    CFG.update(T=64, B=64); print("V =", V, "| ln V =", round(math.log(V), 3), "|", len(real), "characters")
    torch.manual_seed(0); rb = train_model(Bigram(V).to(DEVICE), steps=2000, log_every=500)
    torch.manual_seed(0); rm = train_model(LM(V, 64, 64).to(DEVICE), steps=3000, log_every=500)
    print(f"real text: bigram {eval_loss(rb):.3f}   one head {eval_loss(rm):.3f}   ln V {math.log(V):.3f}")
    print(generate(rm, n=300, T=64))

In [ ]:
#@title ✍️ F1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("F1", what_happened, surprise)

---
## Wrap up

Run the cell below, copy the table into your insight ledger next to `core-transformer`, and mark the biggest surprise.
Then on the laptop:

```
python track.py done week-04/build --rating <1-5> --minutes 180 --note "biggest surprise"
python gen_week.py 5
```

In [ ]:
reveal()